# Artists

This notebook shows work done to derive the complete list of all artists in the Sounds of San Antonio data set.

- [Load and Inspect the Data](#load-and-inspect-the-data)
- [Get the List of Unique Artists](#get-the-list-of-unique-artists)
- [Investigate Artists Outside the Master Concert List](#investigate-artists-outside-the-master-concert-list)
- [Add ISO Date for Concerts](#add-iso-date-for-concerts)
- [Create Table With First and Last Artist Apperances](#create-table-with-first-and-last-artist-apperances)
- [Create Table With Unique Indexes for All Artists](#create-table-with-unique-indexes-for-all-artists)
- [Write Table to Excel](#write-table-to-excel)

There are several lists of artists to work with:

1. The `artistsIndex` worksheet. This sheet has 1769 rows. Each row has two columns, an artist name and a unique index for that artist. This is the model we want to follow for the basic Sounds of San Anto data: each bit of atomic data should have a unique identifier. We want all artists to have a unique artist index.
2. The `Venue_Artist_Event` worksheet also has artists. This is the master list of all concerts. The sheet has columns for the venue, the artist, the full list of artists at the event, the name of the event, and a month, day, and year. Each concert also has a unique index.
3. The `Ven_Ev_Ar_Address` worksheet has artists listed. This is a derived sheet that contains addresses and coordinates for events from 1970 to 2009. The expectation is that all artists in this sheet are also present in `Venue_Artist_Event` since this sheet was derived from that one.

### Load and Inspect the Data

I'm going to load the Sounds of San Anto Excel file, then inspect the names of the worksheets within the file. This way, I will be able to access each worksheet individually.

In [1]:
import pandas as pd
import numpy as np
import datetime
from dateutil import parser
import os

In [2]:
sounds_data = pd.ExcelFile('input-data//product1_data_master_rev.xlsx')
sheets = sounds_data.sheet_names
sheets

['Ven_Ev_Ar_Address',
 'Venue_Artist_Event',
 'Layer 1 Venues',
 'artistsGenre',
 'genreIndex',
 'Sheet5',
 'artistsIndex',
 'Venues No Address',
 'venues_with_coordinates']

Now, I can examine each worksheet I'm interested in, individually.

In [3]:
artists_index = pd.read_excel('input-data//product1_data_master_rev.xlsx', sheet_name='artistsIndex')
venue_artist_event = pd.read_excel('input-data//product1_data_master_rev.xlsx', sheet_name='Venue_Artist_Event')
ven_ev_ar_address = pd.read_excel('input-data//product1_data_master_rev.xlsx', sheet_name='Ven_Ev_Ar_Address')

First, I examine the artistsIndex data. I see there are 1769 entries, but only 1768 non-null rows. I examine the nature of the data by looking at the first few rows. Then I find the entry that has a null name. I find that row 820, with the index 821, has no name. We'll want to drop that row before constructing the final list.

In [4]:
artists_index.info()

<class 'pandas.DataFrame'>
RangeIndex: 1769 entries, 0 to 1768
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Name    1768 non-null   str  
 1   Index   1769 non-null   int64
dtypes: int64(1), str(1)
memory usage: 27.8 KB


In [5]:
artists_index.head()

,Name,Index
0,Dierks Bentley,1
1,"""The Italian Cowboy""",2
2,Keith Urban,3
3,George Jones,4
4,Hilary Duff,5


In [6]:
artists_index[artists_index.isnull().any(axis=1)]

,Name,Index
820,NaN,821


In [7]:
artists_index = artists_index.dropna()

In [8]:
artists_index.info()

<class 'pandas.DataFrame'>
Index: 1768 entries, 0 to 1768
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Name    1768 non-null   str  
 1   Index   1768 non-null   int64
dtypes: int64(1), str(1)
memory usage: 41.4 KB


Now I'm going to investigate potential duplicates. I don't find exact duplicates, but if I use the `.strip()` string method to elimininate surrounding whitespace before doing the comparison, I do find four duplicate rows. Investigating further, I find that Tim McGraw and Angerkill are mentioned twice. Further, there are duplicate entries for both SA Symphony and S.A. Symphony. It would be good to settle on a spelling of this artist entry, but we'll want to see if the same duplication is present also in the other sheets.

In [9]:
artists_index[artists_index['Name'].str.strip().duplicated()]

,Name,Index
328,Tim McGraw,329
485,Angerkill,486
595,SA Symphony,596
1164,S.A. Symphony,1165


In [10]:
artists_index[artists_index['Name'].str.contains('Tim Mc')]

,Name,Index
25,Tim McGraw,26
328,Tim McGraw,329


In [11]:
artists_index[artists_index['Name'].str.contains('Anger')]

,Name,Index
483,Angerkill,484
485,Angerkill,486


In [12]:
artists_index[artists_index['Name'].str.contains('SA Symphony')]

,Name,Index
570,SA Symphony,571
595,SA Symphony,596


In [13]:
artists_index[artists_index['Name'].str.contains('S.A. Symphony')]

,Name,Index
1127,S.A. Symphony,1128
1164,S.A. Symphony,1165


Next, I'll look at the `Venue_Artist_Event` sheet. I see here there are 37892 entries. The artist name is in the column called `Artist_Formula`. There are no nulls in this list. Howewver, we can expect that there are many duplicate artist names, since many artists had more than one concert.

In [14]:
venue_artist_event.info()

<class 'pandas.DataFrame'>
RangeIndex: 37892 entries, 0 to 37891
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Month           37892 non-null  str  
 1   Day             37892 non-null  int64
 2   Year            37892 non-null  int64
 3   Venue           37892 non-null  str  
 4   Event_Artists   37892 non-null  str  
 5   Artist_Formula  37892 non-null  str  
 6   Event_Formulas  7723 non-null   str  
 7   Index           37892 non-null  int64
dtypes: int64(3), str(5)
memory usage: 2.3 MB


In [15]:
venue_artist_event.head()

,Month,Day,Year,Venue,Event_Artists,Artist_Formula,Event_Formulas,Index
0,September,10,1857,St. Mary's Catholic Church,French Mountaineer Singers,French Mountaineer Singers,NaN,1
1,June,25,1869,Menger Hotel,General Mason Farewell Party: San Antonio Bras...,San Antonio Brass Band,General Mason Farewell Party,2
2,July,4,1874,San Pedro Park,San Antonio Brass Band,San Antonio Brass Band,NaN,3
3,October,9,1874,"San Pedro Springs, Turner Hall",Tenth Texas Saengerfest,Tenth Texas Saengerfest,NaN,4
4,October,10,1874,"Casino Hall, Turner Hall",Tenth Texas Saengerfest,Tenth Texas Saengerfest,NaN,5


I'm going to investigate the issue with variants of San Antonio Symphony. I filter the data set to include just the rows in which the artist name contains the word 'Symphony', then get just the artist name column, and then get just the unique values. I inspect them visually and find that this sheet doesn't include 'S.A. Symphony' or 'SA Symphony' at all. It only uses the full name 'San Antonio Symphony' (as well as 'San Antonio Symphony Orchestra').

This means the actual concert list doesn't include some of the artist names in the artistsIndex sheet. We could drop both 'S.A. Symphony' and 'SA Symphony' from the artist list and include just the full term 'San Antonio Symphony'. I won't judge whether 'San Antonio Symphony' and 'San Antonio Symphony Orchestra' are the same thing.

In [16]:
all_symphonies = venue_artist_event[venue_artist_event['Artist_Formula'].str.contains('Symphony')]['Artist_Formula'].unique()
for symphony in all_symphonies:
    print(symphony)

Symphony Orchestra
New York Symphony Orchestra conducted by Walter Damrosch
Minneapolis Symphony Orchestra
San Antonio Symphony Orchestra featuring Elllison Van Hoose (Julien Paul Blitz's first as conductor)
Kelly Field Symphony Orchestra
San Antonio Symphony Orchestra featuring Francisco Hernandez (violin)
Symphony Orchestra featuring Rafealo Diaz (tenor)
Symphony Orchestra (Flora Briggs pianist
Don Felice and the Palace Symphony Orchestra
Royal Theater Symphony Orcestra featuring Helen E. Smith
Don Felice and the Palace Symphony Orchestra featuring Jose Tovar (piano)
Aztec Symphony Orchestra
Bandera Little Symphony Orchestra
St. Louis Symphony Orchestra
Federal Symphony Orchestra
San Antonio Federal Symphony Orchestra
San Antonio Symphony Orchestra featuring Alec Templeton
San Antonio Symphony Orchestra featuring Rose Bampton (soprano)
San Antonio Symphony Orchestra featuring Richard Crooks
San Antonio Symphony Orchestra with Carlos Chaves guest conductor
San Antonio Symphony Orchest

## Get the List of Unique Artists

Now I'll get the list of unique artists in the `venue_artist_event` sheet. There are 20320 unique artists.

In [17]:
vae_artists = venue_artist_event['Artist_Formula'].dropna().str.strip().unique()
vae_artists

<StringArray>
[                      'French Mountaineer Singers',
                           'San Antonio Brass Band',
                          'Tenth Texas Saengerfest',
 'Eighth Cavalry Band (conducted by Frank P. Hall)',
         'Fay Templeton and the Star Opera Company',
                               'Charlotte Thompson',
                              'Eighth Cavalry Band',
                                  'Lillian Spencer',
                                   'Angela Peralta',
                                     'Katie Putnam',
 ...
                                        'Sam Evian',
                                       'The Struts',
                                       'Mac Saturn',
                                    'Hot for Crime',
                                     'Steve Oliver',
                                  'Jeff Rosenstock',
                                      'Small Crush',
                      'Lil Darkie and the Collapse',
                           

My expectation is that the artists in `Ven_Ev_Ar_Address` are simply a subset of the ones in `Venue_Artist_Event`. I'll see if that's true now. First I'll just take a look at the data, verify the column names, and see if there are any nulls. I don't see any, but just as best practice I'll still drop nulls when I derive the unique artist list.

To derive the unique artist list, I select the 'Artist' column, drop nulls, strip whitespace, and get the unique entries. There are 11009 artists in that sheet. Then I use numpy to concatenate the two lists. I use the `set` operation to reduce the list to unique entries and find that the length of the set is 20320, which is exactly the same as the length of the list for just `Venue_Artist_Event`. This means I was right in assuming that the artists in `Ven_Ev_Ar_Address` are a subset of `Venue_Artist_Event`. 

I conclude that we don't need to include `Ven_Ev_Ar_Address` in deriving the complete artist list.

In [18]:
ven_ev_ar_address.info()

<class 'pandas.DataFrame'>
RangeIndex: 20638 entries, 0 to 20637
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          20638 non-null  int64  
 1   Month          20638 non-null  str    
 2   Day            20638 non-null  int64  
 3   Year           20638 non-null  int64  
 4   Venue          20638 non-null  str    
 5   Event_Artists  20638 non-null  str    
 6   Address        17542 non-null  str    
 7   Zip            17540 non-null  float64
 8   Longitude      6446 non-null   float64
 9   Latitude       6446 non-null   float64
 10  Artist         20638 non-null  str    
 11  Event          4503 non-null   str    
 12  City           17540 non-null  str    
 13  State          17542 non-null  str    
dtypes: float64(3), int64(3), str(8)
memory usage: 2.2 MB


In [19]:
veaa_artists = ven_ev_ar_address['Artist'].dropna().str.strip().unique()
veaa_artists

<StringArray>
[                          'Willie Nelson',
                           'Charley Pride',
                           'Grandpa Jones',
                           'David Houston',
                              'Vikki Carr',
               'Chicago Transit Authority',
                             'Youngbloods',
                             'James Brown',
 'Woody Herman and his 16 piece orchestra',
          'Johnny Bush and the Bandoleros',
 ...
                        'Billy Currington',
                          'Sons of Sylvia',
                'UTSA Percussion Ensemble',
                           'Justin Bieber',
                            'Mark Stewart',
                            'Jason Boland',
                              'Jason Eady',
                           'Cadilac Ranch',
                'Baroque Ensemble Retablo',
              'Cody Canada & The Departed']
Length: 11009, dtype: str

In [20]:
np.concatenate((veaa_artists, vae_artists))

array(['Willie Nelson', 'Charley Pride', 'Grandpa Jones', ...,
       'Lil Darkie and the Collapse', 'Walter Beasley',
       'Parliament Funkadelic featuring George Clinton'],
      shape=(31329,), dtype=object)

In [21]:
len(set(np.concatenate((veaa_artists, vae_artists))))

20320

## Investigate Artists Outside the Master Concert List

A separate question is: if there is an artist in `artistsIndex` that is not in `Venue_Artist_Event`, should they be included in the final artist list? The examples are "SA Symphony" and "S.A. Symphony". We know these are not in the artist column in the concert list. But if they don't have associated concerts, should they be in the final list? I would argue no, because the basic data set is a data set on concerts. If we have some artists in a sheet that have no concerts in the concert set, I don't know where that came from, but they are not part of the concert data set, therefore they are not part of the Sounds of San Anto data.

In conclusion, I will just include these 20320 artists in the final artist list.

## Add ISO Date for Concerts

I would like the artists to be sortable in some way by the dates in which they appear in the data. That is, I would like it to be possible to sort them chronologically as well as alphabetically. 

To accomplish this, I first will add a column with an ISO-formatted date for each concert.

In [22]:
def str_date_to_iso(month, day, year):
    str_date = f"{month} {day} {year}"
    dt = parser.parse(str_date).date().isoformat()
    return dt

In [23]:
venue_artist_event["date"] = venue_artist_event.apply(lambda row: str_date_to_iso(row['Month'], row['Day'], row['Year']), axis=1)

In [24]:
venue_artist_event.head()

,Month,Day,Year,Venue,Event_Artists,Artist_Formula,Event_Formulas,Index,date
0,September,10,1857,St. Mary's Catholic Church,French Mountaineer Singers,French Mountaineer Singers,NaN,1,1857-09-10
1,June,25,1869,Menger Hotel,General Mason Farewell Party: San Antonio Bras...,San Antonio Brass Band,General Mason Farewell Party,2,1869-06-25
2,July,4,1874,San Pedro Park,San Antonio Brass Band,San Antonio Brass Band,NaN,3,1874-07-04
3,October,9,1874,"San Pedro Springs, Turner Hall",Tenth Texas Saengerfest,Tenth Texas Saengerfest,NaN,4,1874-10-09
4,October,10,1874,"Casino Hall, Turner Hall",Tenth Texas Saengerfest,Tenth Texas Saengerfest,NaN,5,1874-10-10


## Create Table With First and Last Artist Apperances

Now, I'll create a new table (data frame) that has each artist listed with the dates in which they appear for the first time and the last time. I want to make sure I don't have duplicates based on leading or trailing white space, which as we have seen is an issue in the data. So I apply the `str.strip()` method first. Then I group the data by artist, select the date column, and create min and max columns for each artist based on the dates.

Just as a sanity check, I also take a peek at what artists actually have a difference between their first and last dates of appearance. There are 5605 artists who have more than one concert to their name.

In [25]:
venue_artist_event["Artist_Formula"] = venue_artist_event["Artist_Formula"].str.strip()

venue_artist_event_dates = (
    venue_artist_event
    .groupby("Artist_Formula")["date"]
    .agg(["min", "max"])
    .reset_index()
)

In [26]:
venue_artist_event_dates

,Artist_Formula,min,max
0,!!! Chk Chk Chk,2023-10-12,2023-10-12
1,$Not,2023-05-09,2023-05-09
2,'Blue' Gene Tyranny,1985-06-28,1985-06-28
3,'Gate Mouth' Brown Blues As You Like Them,1947-01-31,1947-02-01
4,.38 Special,1984-04-01,2009-06-02
...,...,...,...
20315,vocals],2010-02-27,2010-02-27
20316,west,1986-02-08,1986-02-08
20317,with added attraction Curley Mays,1958-03-06,1958-03-06
20318,xxxx,1996-06-08,1996-06-08


In [27]:
# artists for whom the date of first appearance is not the same as the date of last appearance

venue_artist_event_dates[venue_artist_event_dates["min"] != venue_artist_event_dates["max"]]

,Artist_Formula,min,max
3,'Gate Mouth' Brown Blues As You Like Them,1947-01-31,1947-02-01
4,.38 Special,1984-04-01,2009-06-02
6,1,1997-08-31,1998-12-25
9,10 Years,2008-04-25,2013-03-15
12,112,1999-06-26,2001-09-19
...,...,...,...
20290,the Westernairs,1961-12-16,1962-02-17
20291,the Whoosits,1988-09-03,2011-07-09
20292,the Wilburn Brothers,2005-12-04,2005-12-23
20298,the fun boys,1958-01-31,1958-02-07


## Create Table With Unique Indexes for All Artists

Now I'm going to assemble a data frame that has unique indexes for all artists, as well as first and last appearance dates. I am not going to use the artist list in `artistsIndex` as a source of artist names, but I do want to retain any indexes that were assigned in that list. This is so that, if those indexes are in use in any current application, there is no change to the existing indexes.

I'm going to start by renaming the columns in my current data frame, to be a little more clear.

In [28]:
final_artist_list = venue_artist_event_dates.rename(columns={"Artist_Formula": "Artist", "min": "First Appearance", "max": "Last Appearance"})

In [29]:
final_artist_list.head()

,Artist,First Appearance,Last Appearance
0,!!! Chk Chk Chk,2023-10-12,2023-10-12
1,$Not,2023-05-09,2023-05-09
2,'Blue' Gene Tyranny,1985-06-28,1985-06-28
3,'Gate Mouth' Brown Blues As You Like Them,1947-01-31,1947-02-01
4,.38 Special,1984-04-01,2009-06-02


Next, I'm going to use the pandas data frame `.merge()` method to join together the artists index data frame with the table that contains all the artists along with the first and last appearance dates. This merge operation looks for matches between the "Name" column in one data frame and the "Artist" column in the other one.

I can take a look at the result and see that wherever it found a match, which is in 1665 rows, it added the index. Of course that means there are over 18 thousand rows that have no index yet. It assigned the value `NaN` to these rows. Since `NaN` is technically a floating point value, it also transformed indexes into decimals. We'll fix that later.

In [30]:
final_artist_list = final_artist_list.merge(
    artists_index[["Name", "Index"]],
    left_on="Artist",
    right_on="Name",
    how="left"
)

In [31]:
final_artist_list[final_artist_list["Name"].notnull()]

,Artist,First Appearance,Last Appearance,Name,Index
9,10 Years,2008-04-25,2013-03-15,10 Years,1619.0
12,112,1999-06-26,2001-09-19,112,364.0
13,12 Step Rebels,2004-02-27,2004-02-27,12 Step Rebels,936.0
18,13th Floor Elevators,1973-07-04,1973-07-04,13th Floor Elevators,1095.0
34,214th Army Band,1977-07-27,1977-07-27,214th Army Band,392.0
...,...,...,...,...,...
20283,the True Believers,1985-06-10,1985-06-10,the True Believers,448.0
20284,the Tyrones,1988-10-01,1991-09-28,the Tyrones,1680.0
20314,vocals; Chris Wolfe,2007-08-18,2007-08-18,vocals; Chris Wolfe,482.0
20315,vocals],2010-02-27,2010-02-27,vocals],537.0


In [32]:
final_artist_list

,Artist,First Appearance,Last Appearance,Name,Index
0,!!! Chk Chk Chk,2023-10-12,2023-10-12,NaN,NaN
1,$Not,2023-05-09,2023-05-09,NaN,NaN
2,'Blue' Gene Tyranny,1985-06-28,1985-06-28,NaN,NaN
3,'Gate Mouth' Brown Blues As You Like Them,1947-01-31,1947-02-01,NaN,NaN
4,.38 Special,1984-04-01,2009-06-02,NaN,NaN
...,...,...,...,...,...
20315,vocals],2010-02-27,2010-02-27,vocals],537.0
20316,west,1986-02-08,1986-02-08,NaN,NaN
20317,with added attraction Curley Mays,1958-03-06,1958-03-06,NaN,NaN
20318,xxxx,1996-06-08,1996-06-08,xxxx,229.0


The "Name" column is not useful to us any more, so we'll drop it.

In [33]:
final_artist_list = final_artist_list.drop(columns=['Name'])

In [34]:
final_artist_list

,Artist,First Appearance,Last Appearance,Index
0,!!! Chk Chk Chk,2023-10-12,2023-10-12,NaN
1,$Not,2023-05-09,2023-05-09,NaN
2,'Blue' Gene Tyranny,1985-06-28,1985-06-28,NaN
3,'Gate Mouth' Brown Blues As You Like Them,1947-01-31,1947-02-01,NaN
4,.38 Special,1984-04-01,2009-06-02,NaN
...,...,...,...,...
20315,vocals],2010-02-27,2010-02-27,537.0
20316,west,1986-02-08,1986-02-08,NaN
20317,with added attraction Curley Mays,1958-03-06,1958-03-06,NaN
20318,xxxx,1996-06-08,1996-06-08,229.0


There is another source of unique artist indexes. In late 2025 and early 2026 Jacob Sherman worked to associate artists with genres. As he added artist-genre identifications, he gave the artists new indexes starting with 1770. As with the original artist indexes, I would like to retain these existing indexes, so I will now add them to the table. 

First, I'll load the data I want to add, and inspect the two tables that I will be using. I want to transfer artist Indexes from `artists_genres_new` to `final_artist_list`.

In [35]:
artists_genres_new = pd.read_excel('input-data/artists_genres_new.xlsx')

In [36]:
artists_genres_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 10102 entries, 0 to 10101
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Name                     10101 non-null  str    
 1   Sheet1.name variation    261 non-null    str    
 2   Genres                   10096 non-null  str    
 3   artistIndex              10102 non-null  int64  
 4   Artists_Genre (2).Index  4791 non-null   float64
 5   Name2                    5305 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 473.7 KB


In [37]:
final_artist_list.info()

<class 'pandas.DataFrame'>
RangeIndex: 20320 entries, 0 to 20319
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Artist            20320 non-null  str    
 1   First Appearance  20320 non-null  str    
 2   Last Appearance   20320 non-null  str    
 3   Index             1665 non-null   float64
dtypes: float64(1), str(3)
memory usage: 635.1 KB


The strategy here will be:
1. Read indexes from `artists_genres_new` into a new lookup data frame that connects names and indexes.
2. Create a boolean mask to mark artists in `final_artist_list` that don't have indexes yet.
3. Take all the artists that don't have indexes yet in `final_artist_list`, and assign the values from the lookup data frame to the index column in `final_artist_list`.

In [41]:
name_index_lookup = (
    artists_genres_new.dropna(subset=['Name', 'artistIndex'])
    .drop_duplicates(subset='Name')
    .set_index('Name')['artistIndex']
)

no_indexes_yet = final_artist_list['Index'].isna()

final_artist_list.loc[no_indexes_yet, 'Index'] = final_artist_list.loc[no_indexes_yet, 'Artist'].map(name_index_lookup)

In [42]:
final_artist_list.info()

<class 'pandas.DataFrame'>
RangeIndex: 20320 entries, 0 to 20319
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Artist            20320 non-null  str    
 1   First Appearance  20320 non-null  str    
 2   Last Appearance   20320 non-null  str    
 3   Index             6265 non-null   float64
dtypes: float64(1), str(3)
memory usage: 635.1 KB


Here comes the heart of the operation. I'll create a "next index" variable, which is the highest index we've assigned yet. That would be the in the `artists_genres_new` data frame plus one. This is where we want to start our indexing for any artists that don't have indexes.

Next, I'll create a boolean mask, which is basically a series of "True" and "False" values that identify where the index is missing. To do this I use the `.isna()` method. Then I count how many rows are missing indexes, using the `.sum()` method. This is kind of strange, but since True is actually the value 1, and False is actually the value 0, you can add all the True values using .sum(). 

The final bit is an assignment of values between iterables with equal counts. On the left, we get the pandas indexes (that is the row numbers) for all the rows in the "Index" column that are missing values. On the right we create a `range` object based on the total number of indexes that are missing. We know the numbers on both sides will match. Because of this, pandas will know to assign the values on the right to the locations on the left.

We do get indexes up to 20424 for 20320 rows. This is because we know that there are artists in `artistsIndex` that had assigned indexes but are not in `venue_artist_event`. These assigned indexes were not reused, because you are not supposed to go back and fill in unused indexes in a data set, even if the row is deleted and the index disappears. You always start with the largest index plus one and keep counting from there. As a result we have some numerical indexes that are not being used, about 105 or so.

In [44]:
next_index = artists_genres_new['artistIndex'].max() + 1
missing_index = final_artist_list["Index"].isna()
total_missing_index = missing_index.sum()
final_artist_list.loc[missing_index, "Index"] = range(next_index, next_index + total_missing_index)

In [45]:
final_artist_list

,Artist,First Appearance,Last Appearance,Index
0,!!! Chk Chk Chk,2023-10-12,2023-10-12,6388.0
1,$Not,2023-05-09,2023-05-09,6389.0
2,'Blue' Gene Tyranny,1985-06-28,1985-06-28,6390.0
3,'Gate Mouth' Brown Blues As You Like Them,1947-01-31,1947-02-01,6391.0
4,.38 Special,1984-04-01,2009-06-02,3719.0
...,...,...,...,...
20315,vocals],2010-02-27,2010-02-27,537.0
20316,west,1986-02-08,1986-02-08,3268.0
20317,with added attraction Curley Mays,1958-03-06,1958-03-06,20441.0
20318,xxxx,1996-06-08,1996-06-08,229.0


Now that all the rows are filled in the Index column, and we don't have any `NaN` values, we can redefine the Index column as an integer column. We now have a table with all the artists in our data set, their dates of first and last appearance, and a unique identifier for all of them.

I'll also put the Index column first.

In [46]:
final_artist_list['Index'] = final_artist_list['Index'].astype(int)

In [47]:
final_artist_list = final_artist_list[['Index', 'Artist', 'First Appearance', 'Last Appearance']]
final_artist_list

,Index,Artist,First Appearance,Last Appearance
0,6388,!!! Chk Chk Chk,2023-10-12,2023-10-12
1,6389,$Not,2023-05-09,2023-05-09
2,6390,'Blue' Gene Tyranny,1985-06-28,1985-06-28
3,6391,'Gate Mouth' Brown Blues As You Like Them,1947-01-31,1947-02-01
4,3719,.38 Special,1984-04-01,2009-06-02
...,...,...,...,...
20315,537,vocals],2010-02-27,2010-02-27
20316,3268,west,1986-02-08,1986-02-08
20317,20441,with added attraction Curley Mays,1958-03-06,1958-03-06
20318,229,xxxx,1996-06-08,1996-06-08


## Write Table to Excel

It's time to write the resulting data frame to Excel. I generate a reusable method that will create the file if it doesn't exist, but just create or replace the sheet if the file already exists.

In [48]:
def write_df_to_excel(df, filepath, sheet_name):
    """Write a DataFrame to a sheet in an Excel file, creating the file
    if it doesn't exist, or replacing the sheet if it does."""
    
    mode = "a" if os.path.exists(filepath) else "w"
    kwargs = {"if_sheet_exists": "replace"} if mode == "a" else {}

    with pd.ExcelWriter(filepath, engine="openpyxl", mode=mode, **kwargs) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

In [50]:
# write_df_to_excel(final_artist_list, 'output-data/sounds-of-san-anto.xlsx', 'artists')